# Statistical vs ML feature extraction

Recomputes the Part 3 overlap from two catalogues (not a re-run of EDA or the selectors):

- Statistical FDR set (n = 20, time-at-risk excluded) from `eda.ipynb`
- ML consensus set (n = 10, union of within-model LOCO ∩ SHAP ∩ FFS top-20, PR-AUC) from the
  2026-09-19 anti-leakage dump of `baseline_feature_selections.ipynb`
  (`model_feature_selectors_antileak/`). TSSI and WBC are dropped from the ML matrix.

Writes figures and CSVs to `paper_results/03_stats_vs_ml/paper_figures/` only
(via `rebuild_part3_paper_figures.py`). Pre-antileak cached CSVs and the 2026-08-31
ML-13 / Jaccard 5/28 catalogues are superseded — do not resume them.

This is a **methods comparison** of two extraction procedures, not a biological finding.
Jaccard is 5/25 = 0.20.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [HERE, *HERE.parents]
        if (p / "code" / "modeling" / "tools" / "rebuild_part3_paper_figures.py").is_file()
    ),
    HERE,
)
sys.path.insert(0, str(ROOT / "code" / "modeling" / "tools"))
import rebuild_part3_paper_figures as p3  # noqa: E402

print("ROOT:", ROOT)
print("Dump:", p3.DUMP)


ROOT: /home/fadia/Documents/vlst
Dump: /home/fadia/Documents/vlst/code/modeling/interpretability/Kaggle_baseline_intrepretability_results/baseline_interpretability_results/model_feature_selectors_antileak


In [2]:
# FDR names: eda.ipynb (unchanged). ML consensus: live anti-leak dump, not the 2026-08-31 ML-13 list.
STATS_FDR = p3.STATS_FDR
STATS_MULTIVAR = p3.STATS_MULTIVAR
TIME_AT_RISK = p3.TIME_AT_RISK
DOMAINS = p3.DOMAINS
SHARED_ROWS = p3.SHARED_ROWS
STATS_ONLY_WHY = p3.STATS_ONLY_WHY
ML_ONLY_WHY = p3.ML_ONLY_WHY
BUCKET = p3.BUCKET

ML_CONSENSUS = p3.ml_consensus_from_dump()
ML_FREQUENT_EXTRA = p3.ml_frequent_extra(set(ML_CONSENSUS))
print("ML consensus from dump (n=%d):" % len(ML_CONSENSUS))
print("  " + "; ".join(ML_CONSENSUS))
print("Frequent extra (n=%d):" % len(ML_FREQUENT_EXTRA))
print("  " + "; ".join(ML_FREQUENT_EXTRA))


ML consensus from dump (n=10):
  Clopidogrel; Cre; HGB; HbA1c; LDL; LV; Men; No postdilation; Stent type-SES_xiencev; eGFR
Frequent extra (n=10):
  LVEF; Total stent length; Previous PCI; 1.1:1Post dilation; CaI; Stent type-SES_tivoli; Initial diagnosis-AMI; TCL; Fast-Glu; PES


In [3]:
# Figure / CSV writers live in rebuild_part3_paper_figures.py so the notebook
# cannot drift from the dump-backed Part 3 report.
print("Using rebuild_part3_paper_figures writers.")


Using rebuild_part3_paper_figures writers.


In [4]:
stats, ml = set(STATS_FDR), set(ML_CONSENSUS)
inter = stats & ml
union = stats | ml
assert len(STATS_FDR) == 20
assert len(ML_CONSENSUS) == 10, ML_CONSENSUS
assert inter == {"Clopidogrel", "HbA1c", "LV", "No postdilation", "eGFR"}, inter
assert len(inter) == 5
assert len(union) == 25
assert "WBC" not in ml
jacc = len(inter) / len(union)
print("Catalogues: EDA FDR-20 and 2026-09-19 Part 2 PR-AUC three-way ML-10 (anti-leakage dump).")
print("Part 5 TabPFN ranking is a different extractor and is not an input here.")
print("Pre-antileak ML-13 / Jaccard 5/28 is superseded.")
print()
print("Statistical FDR (n=20):")
print("  " + "; ".join(STATS_FDR))
print()
print("ML consensus LOCO ∩ SHAP ∩ FFS top-20, any classic model (n=10):")
print("  " + "; ".join(ML_CONSENSUS))
print()
print("Intersection (n=5):", ", ".join(sorted(inter)))
print("Stats-only (n=%d):" % len(stats - ml), ", ".join(sorted(stats - ml)))
print("ML-only (n=%d):" % len(ml - stats), ", ".join(sorted(ml - stats)))
print(
    f"stats={len(stats)} ml={len(ml)} intersection={len(inter)} "
    f"union={len(union)} Jaccard={jacc:.4f} (=5/25)"
)
print()
p3.main()


Catalogues: EDA FDR-20 and 2026-09-19 Part 2 PR-AUC three-way ML-10 (anti-leakage dump).
Part 5 TabPFN ranking is a different extractor and is not an input here.
Pre-antileak ML-13 / Jaccard 5/28 is superseded.

Statistical FDR (n=20):
  WBC; eGFR; LV; CKD5; No.of stents per lesion; HbA1c; NO.of vessels; Total stent length; Fiberinogen; 1.1:1Post dilation; No postdilation; CKD90; Previous PCI; 3-vessel disease; Clopidogrel; Diabetes; PES; Multi-vessel CAD; Single-vessel disease; Stent type-SES

ML consensus LOCO ∩ SHAP ∩ FFS top-20, any classic model (n=10):
  Clopidogrel; Cre; HGB; HbA1c; LDL; LV; Men; No postdilation; Stent type-SES_xiencev; eGFR

Intersection (n=5): Clopidogrel, HbA1c, LV, No postdilation, eGFR
Stats-only (n=15): 1.1:1Post dilation, 3-vessel disease, CKD5, CKD90, Diabetes, Fiberinogen, Multi-vessel CAD, NO.of vessels, No.of stents per lesion, PES, Previous PCI, Single-vessel disease, Stent type-SES, Total stent length, WBC
ML-only (n=5): Cre, HGB, LDL, Men, Stent ty